In [ ]:
import os, json
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from models.weight_cnn import DASWeightCNN
from dataset import DASWeightDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
DATA_DIR = "data"
SAVE_DIR = "results/weight"

df = pd.read_csv(os.path.join(DATA_DIR, "weight_labels.csv"))
df["data_path"] = df["sample_id"].apply(
    lambda x: f"{DATA_DIR}/denoised/denoised_sample_{str(x).zfill(6)}.npy"
)

with open(os.path.join(SAVE_DIR, "splits.json")) as f:
    splits = json.load(f)

test_df = df[df["sample_id"].isin(splits["test"])].reset_index(drop=True)
print(f"Test set: {len(test_df)} samples")
display(test_df[["sample_id", "vehicle_type", "weight"]].head(10))

In [ ]:
test_ds = DASWeightDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)
print(f"Test samples: {len(test_ds)}")

In [ ]:
sample_x, _ = test_ds[0]
num_spatial_channels = sample_x.shape[1]

model = DASWeightCNN(spatial_channels=num_spatial_channels).to(device)

In [ ]:
# Load the trained model saved by: python train.py --task weight
model.load_state_dict(torch.load("results/weight/best_model.pt", map_location=device))
model.eval()
print("Model loaded from results/weight/best_model.pt")

In [6]:
# ---------------------------------------------------------
# Cell 6: Evaluation (Restoring Actual Pounds)
# ---------------------------------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error

model.eval()
actuals, predictions = [], []

# --- In Cell 6, update the evaluation loop ---
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x)
        
# --- In Cell 6, update the evaluation un-scaling ---
        # Reverse the division!
        actual_lbs = y.numpy().flatten() * 1000.0
        raw_pred_lbs = preds.cpu().numpy().flatten() * 1000.0
        
        # Snap negatives to 0
        pred_lbs = np.clip(raw_pred_lbs, a_min=0, a_max=None)
        
        actuals.extend(actual_lbs.tolist())
        predictions.extend(pred_lbs.tolist())

# ... (The rest of Cell 6 remains the same)


actuals_arr = np.array(actuals)
preds_arr = np.array(predictions)

print("--- Test Set Predictions (in lbs) ---")
for act, pred in zip(actuals, predictions):
    print(f"Actual: {act:7.0f} lbs | Predicted: {pred:7.0f} lbs | Error: {abs(act-pred):5.0f} lbs")

mae = mean_absolute_error(actuals_arr, preds_arr)
rmse = np.sqrt(mean_squared_error(actuals_arr, preds_arr))

print("\n--- Final Weight Metrics ---")
print(f"Mean Absolute Error:   {mae:.0f} lbs")
print(f"Root Mean Sq. Error:   {rmse:.0f} lbs")

--- Test Set Predictions (in lbs) ---
Actual:       0 lbs | Predicted:       4 lbs | Error:     4 lbs
Actual:    5325 lbs | Predicted:    3460 lbs | Error:  1865 lbs
Actual:    5150 lbs | Predicted:    4369 lbs | Error:   781 lbs
Actual:    3425 lbs | Predicted:    2993 lbs | Error:   432 lbs
Actual:       0 lbs | Predicted:       6 lbs | Error:     6 lbs
Actual:    4515 lbs | Predicted:    3614 lbs | Error:   901 lbs
Actual:       0 lbs | Predicted:       5 lbs | Error:     5 lbs
Actual:       0 lbs | Predicted:      13 lbs | Error:    13 lbs
Actual:       0 lbs | Predicted:       6 lbs | Error:     6 lbs
Actual:       0 lbs | Predicted:       6 lbs | Error:     6 lbs
Actual:   29000 lbs | Predicted:   25475 lbs | Error:  3525 lbs

--- Final Weight Metrics ---
Mean Absolute Error:   686 lbs
Root Mean Sq. Error:   1262 lbs


In [ ]:
from Utilities import plot_das_data
from config import DAS_FILE
from DAS import DAS
import matplotlib.pyplot as plt

das = DAS(DAS_FILE)
dx = das.meta['dx']
dt = das.meta['dt']

SAMPLE_IDX = 0  # change to inspect different test samples

x_tensor, y_tensor = test_ds[SAMPLE_IDX]
with torch.no_grad():
    raw_pred = model(x_tensor.unsqueeze(0).to(device))

actual_lbs = y_tensor.item() * 1000.0
pred_lbs   = max(0.0, raw_pred.item() * 1000.0)

plot_data = x_tensor.squeeze().numpy()
fig, ax = plt.subplots(figsize=(12, 6))
plot_das_data(data=plot_data, channels=np.arange(plot_data.shape[0]), dx=dx, dt=dt,
              title=f"Actual: {actual_lbs:,.0f} lbs | Predicted: {pred_lbs:,.0f} lbs",
              ax=ax, fig=fig, show=False)
plt.tight_layout()
plt.show()